# Analyse exploratoire des données (EDA)

Objectifs :
1. **Distribution des classes** : le dataset est-il déséquilibré ? Quelles sont les cellules rares ?
2. **Inspection visuelle** : différences morphologiques entre les 13 classes (couleur, taille, forme du noyau).
3. **Dimensions des images** : ont-elles toutes la même résolution ?

Le dossier `IMA205-challenge/` doit être placé à la racine du dépôt.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from wbc.config import DataPaths
from wbc.data import load_train_metadata

paths = DataPaths(ROOT / "IMA205-challenge")
df = load_train_metadata(paths)
print(f"{len(df)} images d'entraînement, {df['label'].nunique()} classes")

## 1. Distribution des classes

In [ ]:
order = df["label"].value_counts().index
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x="label", order=order, hue="label", palette="viridis", legend=False)
plt.title("Distribution des 13 classes de globules blancs", fontsize=16)
plt.xlabel("Type de cellule")
plt.ylabel("Nombre d'images")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

(df["label"].value_counts(normalize=True) * 100).round(2)

## 2. Échantillon visuel de chaque classe

In [ ]:
classes = sorted(df["label"].unique())
n_per_class = 5

fig, axes = plt.subplots(len(classes), n_per_class, figsize=(15, 3 * len(classes)))
for i, cls in enumerate(classes):
    sample = df[df["label"] == cls].sample(n_per_class, random_state=42)
    for j, img_id in enumerate(sample["ID"]):
        img = cv2.imread(str(paths.train_dir / img_id))
        ax = axes[i, j]
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.axis("off")
        if j == 0:
            ax.set_title(cls, fontsize=14, fontweight="bold", loc="left")
plt.subplots_adjust(wspace=0.1, hspace=0.1)
plt.show()

## 3. Dimensions des images (échantillon de 1000)

In [ ]:
sample = df.sample(1000, random_state=42)
dims = set()
for img_id in tqdm(sample["ID"], desc="Lecture des dimensions"):
    img = cv2.imread(str(paths.train_dir / img_id))
    if img is not None:
        dims.add(img.shape[:2])

print(f"Dimensions (hauteur, largeur) trouvées : {dims}")

Résultat obtenu : `{(370, 368), (368, 368)}` — les tailles diffèrent légèrement, d'où le
redimensionnement systématique en 224×224 avant le passage dans le réseau.